# 14 — RT-DETR-L + RetinaNet: Kaggle Ensemble Submission

**Purpose:** inference-only ensemble from already-trained 6-epoch checkpoints; **NO training**, **NO pretrained downloads**. Compare single-detector and combined predictions on the official seven-video validation holdout, then export a Kaggle `submission.csv` for the 3,811 test frames. This is **not** full-train RT-DETR retraining: original models here are 3-fold CV models trained on 2/3 of the original training videos. A separate full-train RetinaNet notebook may be running in Colab; it is independent.

**Inputs to attach manually using Kaggle → Add Input → Upload:**
1. Competition dataset `Multiple Object Video Detection — HW2 UCU AI 2026`.
2. Existing **`rtdetr_tuning_essentials.zip`** (upload as Kaggle dataset, any dataset name), containing `lr2e4_fold_{0,1,2}/weights/best.pt`.
3. A ZIP of **`retinanet_v2_tuning/resolution_768`** from Google Drive. It must contain `fold_0/latest.pt`, `fold_1/latest.pt`, `fold_2/latest.pt`. Download the `resolution_768` folder from Drive (Google Drive automatically zips folders) and upload the resulting ZIP to Kaggle. Alternatively upload the three `.pt` as a structured dataset. **Do not upload RetinaNet `focal_gamma_1p5` or RT-DETR baseline for this particular experiment.**

Kaggle commonly unpacks uploaded ZIPs automatically. This notebook handles both unpacked files and remaining `.zip` archives. It never alters checkpoints in `/kaggle/input`.

**Estimated compute:** GPU inference on six heavyweight models across validation and test may be substantial. Set `FOLDS_TO_USE=(0,)` initially for a two-model diagnostic; use `(0,1,2)` for the planned multi-fold ensemble if GPU budget permits. Results of either configuration are compared under the same holdout protocol. Choose based on independent holdout performance, not only Kaggle public score.

## 1 — Setup, packages, and configuration
GPU in Kaggle Settings; Internet may be needed **only to install Ultralytics**, never to download model weights. If your runtime already has a suitable Ultralytics version, you can skip the install line.

In [1]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.204', 'pycocotools'], check=True)

import os, json, gc, time, shutil, zipfile, gzip, hashlib
from pathlib import Path
import numpy as np, pandas as pd
import torch, torchvision
from PIL import Image
from tqdm.auto import tqdm
from torchvision.transforms.functional import pil_to_tensor
from torchvision.models.detection import retinanet_resnet50_fpn_v2
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import RTDETR

DEVICE='cuda:0' if torch.cuda.is_available() else 'cpu'
assert DEVICE.startswith('cuda'), 'Enable GPU in Kaggle settings.'
ROOT=Path('/kaggle/working/rtdetr_retina_ensemble'); ROOT.mkdir(parents=True,exist_ok=True)
CACHE=ROOT/'prediction_cache'; CACHE.mkdir(exist_ok=True)
FOLDS_TO_USE=(0,1,2)   # Set to (0,) for a faster first diagnostic.
assert set(FOLDS_TO_USE).issubset({0,1,2}) and FOLDS_TO_USE
RT_TRIAL='lr2e4'  # AdamW lr0=2e-4, 6 epochs, 640px (3-fold trial)
RET_TRIAL='resolution_768' # SGD lr=0.002, 6 epochs, min768-max1280
MIN_CONF=0.001
TOP_K_PER_SOURCE=300
RT_BATCH=2
print('Device:',torch.cuda.get_device_name(0))
print('Versions: torch',torch.__version__,'torchvision',torchvision.__version__)
print('Folds:',FOLDS_TO_USE)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Device: Tesla T4
Versions: torch 2.11.0+cu130 torchvision 0.26.0+cu130
Folds: (0, 1, 2)


## 2 — Find data and checkpoints (including auto-extracted ZIP inputs)
This is **read-only** for Kaggle datasets; if ZIP content remains compressed, only the selected checkpoint members are extracted to `/kaggle/working`. Stop if any fold is missing or ambiguous.

In [2]:
DATA_CANDIDATES=[p.parent.parent for p in Path('/kaggle/input').rglob('train/annotations_coco.json')
                 if (p.parent.parent/'val/annotations_coco.json').is_file()
                 and (p.parent.parent/'test/images').is_dir()]
assert len(DATA_CANDIDATES)==1, f'Expected one attached competition dataset, found {DATA_CANDIDATES}'
DATA=DATA_CANDIDATES[0]
print('Competition data:', DATA)

unpacked=list(Path('/kaggle/input').rglob('*.pt'))
zip_paths=sorted(Path('/kaggle/input').rglob('*.zip'))
print('Unpacked .pt files:',len(unpacked),'ZIP inputs:',len(zip_paths))
for p in unpacked[:40]: print(' PT',p)
for p in zip_paths[:20]: print(' ZIP',p)

Competition data: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026
Unpacked .pt files: 10 ZIP inputs: 0
 PT /kaggle/input/datasets/margopaduchak/resolution-768/resolution_768/fold_0/latest.pt
 PT /kaggle/input/datasets/margopaduchak/resolution-768/resolution_768/fold_1/latest.pt
 PT /kaggle/input/datasets/margopaduchak/resolution-768/resolution_768/fold_2/latest.pt
 PT /kaggle/input/datasets/margopaduchak/rtdetr-tuning-essentials/runs/lr5e5_fold_0/weights/best.pt
 PT /kaggle/input/datasets/margopaduchak/rtdetr-tuning-essentials/runs/lr2e4_fold_2/weights/best.pt
 PT /kaggle/input/datasets/margopaduchak/rtdetr-tuning-essentials/runs/lr2e4_fold_1/weights/best.pt
 PT /kaggle/input/datasets/margopaduchak/rtdetr-tuning-essentials/runs/lr2e4_fold_0/weights/best.pt
 PT /kaggle/input/datasets/margopaduchak/rtdetr-tuning-essentials/runs/_baseline_external_fold_2/weights/best.pt
 PT /kaggle/input/datasets/margopaduchak/rtdetr-tuning-essentials/runs/_baseline_external_fo

In [3]:
RESTORED=ROOT/'restored_checkpoints'; RESTORED.mkdir(exist_ok=True)

def resolve_checkpoint(architecture,fold):
    assert fold in (0,1,2)
    if architecture=='rt':
        def matches(name):
            parts=Path(name).parts
            return (Path(name).name=='best.pt' and f'{RT_TRIAL}_fold_{fold}' in parts
                    and 'weights' in parts)
        key=f'rt_{RT_TRIAL}_fold_{fold}'
    elif architecture=='retina':
        def matches(name):
            parts=Path(name).parts
            return (Path(name).name=='latest.pt' and RET_TRIAL in parts
                    and f'fold_{fold}' in parts)
        key=f'retina_{RET_TRIAL}_fold_{fold}'
    else: raise ValueError(architecture)
    direct=[p for p in unpacked if matches(str(p))]
    if len(direct)==1:
        return direct[0]
    if len(direct)>1:
        raise RuntimeError(f'Ambiguous {key} checkpoints: {direct}')
    archive_matches=[]
    for zpath in zip_paths:
        with zipfile.ZipFile(zpath) as z:
            members=[name for name in z.namelist() if matches(name) and not name.endswith('/')]
            archive_matches.extend((zpath,name) for name in members)
    assert len(archive_matches)==1, f'{key}: expected exactly one checkpoint, found {archive_matches}'
    zpath,member=archive_matches[0]
    dest=RESTORED/f'{key}.pt'
    if not dest.exists():
        with zipfile.ZipFile(zpath) as z, z.open(member) as src, dest.open('wb') as dst:
            shutil.copyfileobj(src,dst)
    return dest

RT_WEIGHTS={f:resolve_checkpoint('rt',f) for f in FOLDS_TO_USE}
RET_WEIGHTS={f:resolve_checkpoint('retina',f) for f in FOLDS_TO_USE}
for kind,groups in [('RT-DETR',RT_WEIGHTS),('RetinaNet',RET_WEIGHTS)]:
    for fold,p in groups.items():
        print(kind, 'fold',fold,':',p, f'({p.stat().st_size/1e6:.1f} MB)')
        assert p.stat().st_size>1_000_000

RT-DETR fold 0 : /kaggle/input/datasets/margopaduchak/rtdetr-tuning-essentials/runs/lr2e4_fold_0/weights/best.pt (66.2 MB)
RT-DETR fold 1 : /kaggle/input/datasets/margopaduchak/rtdetr-tuning-essentials/runs/lr2e4_fold_1/weights/best.pt (66.2 MB)
RT-DETR fold 2 : /kaggle/input/datasets/margopaduchak/rtdetr-tuning-essentials/runs/lr2e4_fold_2/weights/best.pt (66.2 MB)
RetinaNet fold 0 : /kaggle/input/datasets/margopaduchak/resolution-768/resolution_768/fold_0/latest.pt (291.3 MB)
RetinaNet fold 1 : /kaggle/input/datasets/margopaduchak/resolution-768/resolution_768/fold_1/latest.pt (291.3 MB)
RetinaNet fold 2 : /kaggle/input/datasets/margopaduchak/resolution-768/resolution_768/fold_2/latest.pt (291.3 MB)


## 3 — Build models **without** downloading pretrained weights
RT-DETR uses Ultralytics `RTDETR(checkpoint.pt)`; RetinaNet uses torchvision with exactly the 8-class classification head from `07_RetinaNet_Tuning`. Note label handling: both RT-DETR and RetinaNet predict **0-based classes 0–7**; COCO categories are 1–8. Do not subtract 1 from RetinaNet predictions.

In [4]:
def build_retina():
    m=retinanet_resnet50_fpn_v2(weights=None,weights_backbone=None,
                               min_size=768,max_size=1280,score_thresh=0.001,
                               nms_thresh=0.5,detections_per_img=300,topk_candidates=1000)
    head=m.head.classification_head
    old=head.cls_logits
    n=m.anchor_generator.num_anchors_per_location()
    assert len(set(n))==1
    head.cls_logits=torch.nn.Conv2d(old.in_channels,n[0]*8,kernel_size=3,stride=1,padding=1)
    head.num_classes=8
    return m

def load_retina(path):
    state=torch.load(path,map_location='cpu',weights_only=True)
    assert isinstance(state,dict) and 'model' in state, f'Unexpected RetinaNet checkpoint format: {path}'
    model=build_retina()
    model.load_state_dict(state['model'],strict=True)
    del state
    return model.eval().to(DEVICE)

# Preflight: independently load exactly one checkpoint of each type; DO NOT train.
m=load_retina(RET_WEIGHTS[FOLDS_TO_USE[0]])
print('RetinaNet checkpoint loaded OK; classes:',m.head.classification_head.num_classes)
del m;gc.collect();torch.cuda.empty_cache()
r=RTDETR(str(RT_WEIGHTS[FOLDS_TO_USE[0]]))
print('RT-DETR checkpoint loaded OK')
del r;gc.collect();torch.cuda.empty_cache()

RetinaNet checkpoint loaded OK; classes: 8
RT-DETR checkpoint loaded OK


## 4 — Independent holdout and test paths
Seven original validation videos are never used in any of these CV checkpoints' training. Evaluate every third holdout frame to control inference time (951 images); Kaggle submission covers **all test images**. For a definitive validation result, change `VAL_STEP=1` before running inference. The COCO evaluation retains original `iscrowd` annotations and `maxDets=[1,10,100]`.

In [6]:

from pathlib import Path
import pandas as pd

VAL_STEP = 3

with (DATA / "val/annotations_coco.json").open() as f:
    val_gt_json = json.load(f)

val_table = pd.DataFrame([
    {
        "id": int(x["id"]),
        "file_name": x["file_name"],
        "clip": Path(x["file_name"]).parent.name,
    }
    for x in val_gt_json["images"]
])

# Deterministic sampling, without pandas groupby.apply warning
val_table = val_table.sort_values(
    ["clip", "file_name"]
).reset_index(drop=True)

val_table["position"] = val_table.groupby(
    "clip"
).cumcount()

val_table = val_table[
    val_table["position"] % VAL_STEP == 0
].copy()

VAL_IDS = val_table["id"].astype(int).tolist()

# Index real validation images
val_image_root = DATA / "val/images"

val_files = sorted(
    p for p in val_image_root.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

val_index = {}

for p in val_files:
    key = (p.parent.name, p.stem)
    if key in val_index:
        raise ValueError(f"Duplicate validation image key: {key}")
    val_index[key] = p

VAL_PATHS = {}
missing = []

for row in val_table.itertuples():
    rel = Path(row.file_name)
    key = (rel.parent.name, rel.stem)

    # First try the exact COCO path
    exact = val_image_root / rel

    if exact.is_file():
        path = exact
    else:
        path = val_index.get(key)

    if path is None:
        missing.append(row.file_name)
    else:
        VAL_PATHS[int(row.id)] = path

print("Validation sampled:", len(VAL_IDS))
print("Validation paths found:", len(VAL_PATHS))
print("Missing:", len(missing))

if missing:
    print("First missing examples:", missing[:10])

assert not missing, "Some validation images could not be located."

# Test images
test_root = DATA / "test/images"

files = sorted(
    p for p in test_root.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

test_path_map = {
    f"{p.parent.name}_{p.stem}": p
    for p in files
}

assert len(test_path_map) == len(files) == 3811, (
    f"Unexpected test image count: {len(files)}"
)

templates = sorted(DATA.rglob("*sample*submission*.csv"))

if templates:
    sample = pd.read_csv(templates[0])
    assert list(sample.columns) == ["image_id", "PredictionString"]
    TEST_IDS = sample["image_id"].astype(str).tolist()
else:
    TEST_IDS = sorted(test_path_map)

assert len(TEST_IDS) == len(set(TEST_IDS))
assert set(TEST_IDS) == set(test_path_map), (
    "Test image IDs do not match sample submission."
)

TEST_PATHS = {k: test_path_map[k] for k in TEST_IDS}

GT = COCO(str(DATA / "val/annotations_coco.json"))

assert {int(c["id"]) for c in GT.dataset["categories"]} == set(range(1, 9))

print(
    "Ready! Holdout:", len(VAL_IDS),
    "images; Test:", len(TEST_IDS),
    "images."
)


Validation sampled: 951
Validation paths found: 951
Missing: 0
loading annotations into memory...
Done (t=0.45s)
creating index...
index created!
Ready! Holdout: 951 images; Test: 3811 images.


## 5 — Run and cache raw predictions (one model at a time)
Each model is loaded separately to reduce GPU memory. Cache files are named by **checkpoint content fingerprint** and split, so changing the model cannot silently reuse old predictions. Current Draft cache is temporary; download the backup in section 9. Prediction coordinates are absolute pixel XYXY.

In [7]:
def fingerprint(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while chunk:=f.read(1024*1024*8): h.update(chunk)
    return h.hexdigest()[:12]

MODELS={**{f'rt_f{f}':('rt',p) for f,p in RT_WEIGHTS.items()},
        **{f'retina_f{f}':('retina',p) for f,p in RET_WEIGHTS.items()}}


def predict_raw(label, paths, split):
    kind,checkpoint=MODELS[label]
    target=CACHE/f'{split}_{label}_{fingerprint(checkpoint)}.json.gz'
    if target.is_file():
        with gzip.open(target,'rt') as f: out=json.load(f)
        assert set(out)==set(map(str,paths)), f'Cache keys mismatch for {target}'
        print('Cached:',target.name)
        return out
    out={}
    if kind=='retina':
        model=load_retina(checkpoint)
        try:
            with torch.inference_mode():
                for image_id,path in tqdm(paths.items(),desc=f'{label} {split}'):
                    with Image.open(path) as img:
                        width,height=img.size
                        tensor=pil_to_tensor(img.convert('RGB')).float().div_(255).to(DEVICE)
                    prediction=model([tensor])[0]
                    boxes=prediction['boxes'].detach().cpu().numpy()
                    scores=prediction['scores'].detach().cpu().numpy()
                    classes=prediction['labels'].detach().cpu().numpy().astype(int)
                    dets=[]
                    for c,s,(x1,y1,x2,y2) in zip(classes,scores,boxes):
                        if not 0<=int(c)<=7 or float(s)<MIN_CONF: continue
                        x1,x2=np.clip([x1,x2],0,width); y1,y2=np.clip([y1,y2],0,height)
                        if x2>x1 and y2>y1: dets.append([int(c),float(s),float(x1),float(y1),float(x2),float(y2)])
                        if len(dets)>=TOP_K_PER_SOURCE: break
                    out[str(image_id)]={'size':[width,height],'detections':dets}
                    del tensor,prediction
        finally:
            del model;gc.collect();torch.cuda.empty_cache()
    elif kind=='rt':
        model=RTDETR(str(checkpoint))
        items=list(paths.items())
        try:
            for start in tqdm(range(0,len(items),RT_BATCH),desc=f'{label} {split}'):
                batch=items[start:start+RT_BATCH]
                results=model.predict(source=[str(p) for _,p in batch],imgsz=640,
                                      device=0,batch=RT_BATCH,conf=MIN_CONF,
                                      iou=0.7,max_det=TOP_K_PER_SOURCE,
                                      verbose=False,save=False,augment=False)
                assert len(results)==len(batch)
                for (image_id,path),res in zip(batch,results):
                    height,width=res.orig_shape
                    dets=[]
                    if res.boxes is not None:
                        for box,c,s in zip(res.boxes.xyxy.cpu().numpy(),
                                         res.boxes.cls.int().cpu().tolist(),
                                         res.boxes.conf.cpu().tolist()):
                            if int(c) not in range(8) or float(s)<MIN_CONF:continue
                            x1,y1,x2,y2=map(float,box)
                            x1,x2=np.clip([x1,x2],0,width);y1,y2=np.clip([y1,y2],0,height)
                            if x2>x1 and y2>y1:dets.append([int(c),float(s),float(x1),float(y1),float(x2),float(y2)])
                    out[str(image_id)]={'size':[width,height],'detections':dets}
        finally:
            del model;gc.collect();torch.cuda.empty_cache()
    else:raise ValueError(kind)
    assert set(out)==set(map(str,paths))
    with gzip.open(target,'wt') as f:json.dump(out,f)
    print('Wrote cache',target,round(target.stat().st_size/1e6,2),'MB')
    return out

print('Models:',list(MODELS))

Models: ['rt_f0', 'rt_f1', 'rt_f2', 'retina_f0', 'retina_f1', 'retina_f2']


In [8]:
# Validation inference (no training). This is the longest validation step.
VAL_RAW={label:predict_raw(label,VAL_PATHS,'val_step3') for label in MODELS}
print('All holdout predictions available.')

rt_f0 val_step3:   0%|          | 0/476 [00:00<?, ?it/s]

Wrote cache /kaggle/working/rtdetr_retina_ensemble/prediction_cache/val_step3_rt_f0_c53068682c1c.json.gz 8.53 MB


rt_f1 val_step3:   0%|          | 0/476 [00:00<?, ?it/s]

Wrote cache /kaggle/working/rtdetr_retina_ensemble/prediction_cache/val_step3_rt_f1_03f854089233.json.gz 8.33 MB


rt_f2 val_step3:   0%|          | 0/476 [00:00<?, ?it/s]

Wrote cache /kaggle/working/rtdetr_retina_ensemble/prediction_cache/val_step3_rt_f2_8b1d49d6e9f3.json.gz 8.17 MB


... (2 outputs left out)

retina_f1 val_step3:   0%|          | 0/951 [00:00<?, ?it/s]

Wrote cache /kaggle/working/rtdetr_retina_ensemble/prediction_cache/val_step3_retina_f1_66948e816d12.json.gz 9.38 MB


retina_f2 val_step3:   0%|          | 0/951 [00:00<?, ?it/s]

Wrote cache /kaggle/working/rtdetr_retina_ensemble/prediction_cache/val_step3_retina_f2_f6e57d887995.json.gz 9.19 MB
All holdout predictions available.


## 6 — Ensemble box fusion and COCOeval
Class-aware weighted fusion: no mixing between object classes; each original model contributes at most one box to each cluster. Select a configuration based on the **same held-out images**. No predictions on test are needed for model selection.

In [9]:
def box_iou(a,b):
    ix1=max(a[0],b[0]);iy1=max(a[1],b[1]);ix2=min(a[2],b[2]);iy2=min(a[3],b[3])
    inter=max(0,ix2-ix1)*max(0,iy2-iy1)
    aa=max(0,a[2]-a[0])*max(0,a[3]-a[1]);bb=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return inter/(aa+bb-inter+1e-9)

def fuse(raw_by_label,image_id,weights,threshold=0.55,max_det=300):
    if not weights:raise ValueError('Empty ensemble')
    image_id=str(image_id)
    sizes={tuple(raw_by_label[l][image_id]['size']) for l in weights}
    assert len(sizes)==1, f'Inconsistent image dimensions: {image_id}: {sizes}'
    total=sum(float(w) for w in weights.values())
    cand={c:[] for c in range(8)}
    for source,w in weights.items():
        for c,s,x1,y1,x2,y2 in raw_by_label[source][image_id]['detections']:
            if s>=MIN_CONF and int(c) in cand:
                cand[int(c)].append((source,float(w),float(s),[x1,y1,x2,y2]))
    final=[]
    for cls,items in cand.items():
        clusters=[]
        for source,w,s,box in sorted(items,key=lambda x:x[1]*x[2],reverse=True):
            ix=-1;best=threshold
            for i,cluster in enumerate(clusters):
                if source in cluster['sources']:continue
                sim=box_iou(box,cluster['box'])
                if sim>best: ix=i;best=sim
            if ix<0:clusters.append({'box':box,'sources':{source},'members':[(w,s,box)]})
            else:
                cl=clusters[ix];cl['sources'].add(source);cl['members'].append((w,s,box))
                cl['box']=np.average([m[2] for m in cl['members']],
                                     weights=[m[0]*m[1] for m in cl['members']],axis=0).tolist()
        for cl in clusters:
            x1,y1,x2,y2=cl['box'];score=sum(w*s for w,s,_ in cl['members'])/total
            if x2>x1 and y2>y1 and score>=MIN_CONF:
                final.append([int(cls),float(score),float(x1),float(y1),float(x2),float(y2)])
    return sorted(final,key=lambda x:x[1],reverse=True)[:max_det]

RT_LABELS=[l for l in MODELS if l.startswith('rt_')]
RET_LABELS=[l for l in MODELS if l.startswith('retina_')]

def balanced_weights(rt_weight,ret_weight):
    return {**{l:rt_weight/len(RT_LABELS) for l in RT_LABELS},
            **{l:ret_weight/len(RET_LABELS) for l in RET_LABELS}}

CANDIDATES={
    'rt_only':{l:1/len(RT_LABELS) for l in RT_LABELS},
    'retina_only':{l:1/len(RET_LABELS) for l in RET_LABELS},
    'equal_ensemble':balanced_weights(1,1),
    'rt_1p5':balanced_weights(1.5,1),
    'retina_1p5':balanced_weights(1,1.5),
}

def coco_detections(raw,ids,weights):
    det=[]
    for img_id in tqdm(ids,desc='Fusing holdout',leave=False):
        for cls,score,x1,y1,x2,y2 in fuse(raw,img_id,weights):
            det.append({'image_id':int(img_id),'category_id':cls+1,
                        'bbox':[x1,y1,x2-x1,y2-y1],'score':score})
    return det

def eval_coco(det,ids):
    assert det,'No detections!'
    dt=GT.loadRes(det)
    ev=COCOeval(GT,dt,iouType='bbox')
    ev.params.imgIds=list(map(int,ids));ev.params.catIds=list(range(1,9));ev.params.maxDets=[1,10,100]
    ev.evaluate();ev.accumulate();ev.summarize()
    s=ev.stats
    return {'mAP50-95':float(s[0]),'AP50':float(s[1]),'AP75':float(s[2]),
            'AP_small':float(s[3]),'AP_medium':float(s[4]),'AP_large':float(s[5])}

rows=[]
for name,weights in CANDIDATES.items():
    print('Evaluating',name)
    det=coco_detections(VAL_RAW,VAL_IDS,weights)
    metrics=eval_coco(det,VAL_IDS)
    rows.append({'configuration':name,'n_predictions':len(det),**metrics})
comparison=pd.DataFrame(rows).sort_values('mAP50-95',ascending=False)
comparison.to_csv(ROOT/'holdout_comparison.csv',index=False)
display(comparison)

Evaluating rt_only


Fusing holdout:   0%|          | 0/951 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=0.38s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=46.70s).
Accumulating evaluation results...
DONE (t=2.12s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.179
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.363
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.152
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.087
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.169
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.318
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.081
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.263
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.361
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

Fusing holdout:   0%|          | 0/951 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=1.10s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=47.23s).
Accumulating evaluation results...
DONE (t=2.14s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.169
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.321
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.153
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.061
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.153
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.318
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.094
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.284
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.354
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

Fusing holdout:   0%|          | 0/951 [00:00<?, ?it/s]

... (2 outputs left out)

Loading and preparing results...
DONE (t=0.38s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=48.33s).
Accumulating evaluation results...
DONE (t=2.20s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.198
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.389
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.174
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.092
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.187
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.344
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.094
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.288
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.395
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

Fusing holdout:   0%|          | 0/951 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=0.39s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=48.98s).
Accumulating evaluation results...
DONE (t=2.16s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.203
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.393
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.183
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.088
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.190
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.350
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.103
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.308
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.415
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

,configuration,n_predictions,mAP50-95,AP50,AP75,AP_small,AP_medium,AP_large
4,retina_1p5,285300,0.202863,0.392505,0.182746,0.088349,0.190424,0.350241
2,equal_ensemble,285300,0.202059,0.393707,0.180748,0.091027,0.190289,0.348952
3,rt_1p5,285300,0.198051,0.389065,0.174193,0.091819,0.186669,0.343684
0,rt_only,285300,0.178901,0.362940,0.151982,0.087482,0.169279,0.318300
1,retina_only,285300,0.168609,0.320986,0.152856,0.060700,0.152892,0.318358


## 7 — Choose candidate and generate Kaggle submission (test inference only)
The best holdout configuration is used by default. **An ensemble may be weaker than either constituent model**; if so, notebook selects the stronger single-model option. Test images are never used for tuning or training.

In [11]:
SELECTED=str(comparison.iloc[0]['configuration'])
assert SELECTED in CANDIDATES
SELECTED_WEIGHTS=CANDIDATES[SELECTED]
print('Selected by holdout COCO mAP:',SELECTED,SELECTED_WEIGHTS)

Selected by holdout COCO mAP: retina_1p5 {'rt_f0': 0.3333333333333333, 'rt_f1': 0.3333333333333333, 'rt_f2': 0.3333333333333333, 'retina_f0': 0.5, 'retina_f1': 0.5, 'retina_f2': 0.5}


In [12]:
# Infer only the required models on all 3811 test images; caching permits cell reruns.
TEST_RAW={label:predict_raw(label,TEST_PATHS,'test_all') for label in SELECTED_WEIGHTS}
print('All test predictions cached.')

rt_f0 test_all:   0%|          | 0/1906 [00:00<?, ?it/s]

Wrote cache /kaggle/working/rtdetr_retina_ensemble/prediction_cache/test_all_rt_f0_c53068682c1c.json.gz 35.57 MB


rt_f1 test_all:   0%|          | 0/1906 [00:00<?, ?it/s]

Wrote cache /kaggle/working/rtdetr_retina_ensemble/prediction_cache/test_all_rt_f1_03f854089233.json.gz 35.34 MB


rt_f2 test_all:   0%|          | 0/1906 [00:00<?, ?it/s]

Wrote cache /kaggle/working/rtdetr_retina_ensemble/prediction_cache/test_all_rt_f2_8b1d49d6e9f3.json.gz 35.34 MB


... (2 outputs left out)

retina_f1 test_all:   0%|          | 0/3811 [00:00<?, ?it/s]

Wrote cache /kaggle/working/rtdetr_retina_ensemble/prediction_cache/test_all_retina_f1_66948e816d12.json.gz 38.64 MB


retina_f2 test_all:   0%|          | 0/3811 [00:00<?, ?it/s]

Wrote cache /kaggle/working/rtdetr_retina_ensemble/prediction_cache/test_all_retina_f2_f6e57d887995.json.gz 38.75 MB
All test predictions cached.


In [13]:
SUBMISSION=ROOT/'submission.csv'
assert not SUBMISSION.exists(), f'{SUBMISSION} exists. Inspect it before regenerating.'
rows=[]
for image_id in tqdm(TEST_IDS,desc='Building submission'):
    fused=fuse(TEST_RAW,image_id,SELECTED_WEIGHTS)
    tokens=[]
    for cls,score,x1,y1,x2,y2 in fused:
        if not(0<=int(cls)<8 and 0<=score<=1 and x2>x1 and y2>y1): continue
        tokens.extend([str(cls),f'{score:.6f}',f'{x1:.2f}',f'{y1:.2f}',f'{x2-x1:.2f}',f'{y2-y1:.2f}'])
    rows.append({'image_id':str(image_id),'PredictionString':' '.join(tokens)})
df=pd.DataFrame(rows)
assert df.columns.tolist()==['image_id','PredictionString']
assert df.image_id.tolist()==TEST_IDS and not df.image_id.duplicated().any()
for pred in df.PredictionString:
    tok=pred.split();assert len(tok)%6==0
    if tok:
        arr=np.asarray(tok,dtype=float).reshape(-1,6)
        assert np.isfinite(arr).all() and np.all(arr[:,0]==np.floor(arr[:,0]))
        assert np.all((arr[:,0]>=0)&(arr[:,0]<8))
        assert np.all((arr[:,1]>=0)&(arr[:,1]<=1))
        assert np.all(arr[:,4:6]>0)
df.to_csv(SUBMISSION,index=False)
settings={'selected':SELECTED,'folds':FOLDS_TO_USE,'weights':SELECTED_WEIGHTS,
          'model_checkpoints':{k:str(p) for k,(_,p) in MODELS.items()},
          'max_dets':300,'iou_for_fusion':0.55,'holdout_sampling_step':VAL_STEP,
          'test_images':len(TEST_IDS)}
(ROOT/'ensemble_settings.json').write_text(json.dumps(settings,indent=2,default=str))
print('READY TO SUBMIT:',SUBMISSION,round(SUBMISSION.stat().st_size/1e6,2),'MB')
display(df.head())

Building submission:   0%|          | 0/3811 [00:00<?, ?it/s]

READY TO SUBMIT: /kaggle/working/rtdetr_retina_ensemble/submission.csv 42.68 MB


,image_id,PredictionString
0,clip_001_0000001,0 0.816140 778.64 886.87 65.43 172.74 0 0.7751...
1,clip_001_0000002,0 0.819426 715.52 896.86 66.11 176.44 0 0.8043...
2,clip_001_0000003,0 0.813465 710.11 898.84 65.51 177.08 0 0.8022...
3,clip_001_0000004,0 0.827384 757.42 902.13 69.26 172.00 0 0.8113...
4,clip_001_0000005,0 0.800102 1663.35 807.07 55.96 153.88 0 0.788...


## 8 — Backup submission and metrics (before leaving Draft)
Creates portable ZIP with predictions, configuration and COCOeval comparison. **Weights are already in uploaded Kaggle datasets** and not duplicated in the backup. If you used ZIPs that were temporary Draft-only uploads, preserve copies of them as well. Do **not** use Save & Run All simply to save your completed interactive run: it would start training/inference cells over again. Download the new ZIP from Kaggle Output while the session remains active.

In [14]:
from zipfile import ZipFile, ZIP_DEFLATED
BACKUP=Path('/kaggle/working/rtdetr_retina_submission_bundle.zip')
required=[ROOT/'submission.csv',ROOT/'holdout_comparison.csv',ROOT/'ensemble_settings.json']
assert all(p.exists() for p in required)
with ZipFile(BACKUP,'w',compression=ZIP_DEFLATED) as z:
    for p in required:z.write(p,arcname=p.name)
with ZipFile(BACKUP) as z:assert z.testzip() is None
print('BACKUP:',BACKUP,round(BACKUP.stat().st_size/1e6,2),'MB')
print('Submission file:',SUBMISSION)
print('Download BOTH files from Output before closing Draft.')

BACKUP: /kaggle/working/rtdetr_retina_submission_bundle.zip 17.56 MB
Submission file: /kaggle/working/rtdetr_retina_ensemble/submission.csv
Download BOTH files from Output before closing Draft.
